# 1. Librerias y conjunto de datos

In [9]:
import os
import requests
import pickle
from joblib import dump
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np
import missingno as msno
from sklearn.model_selection import train_test_split
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from pickle import dump


In [2]:
df = pd.read_csv(r"/workspaces/steven10015-intro-ml/data/raw/url_spam.csv")
df.head(10)

,url,is_spam
0,https://briefingday.us8.list-manage.com/unsubs...,True
1,https://www.hvper.com/,True
2,https://briefingday.com/m/v4n3i4f3,True
3,https://briefingday.com/n/20200618/m#commentform,False
4,https://briefingday.com/fan,True
5,https://www.brookings.edu/interactives/reopeni...,False
6,https://www.reuters.com/investigates/special-r...,False
7,https://www.theatlantic.com/magazine/archive/2...,False
8,https://www.vox.com/2020/6/17/21294680/john-bo...,False
9,https://www.theguardian.com/travel/2020/jun/18...,False


# 2. Preprocesamiento

In [6]:
# Descargas necesarias de NLTK
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

# 1. Función de limpieza y preprocesamiento
def preprocess_url(url):
    # Convertir a minúsculas
    url = url.lower()
    
    # Segmentar la URL: Reemplazar signos de puntuación por espacios
    # Esto separa 'google.com/search' en 'google com search'
    url = re.sub(r'[^\w\s]', ' ', url)
    
    # Tokenización (convertir en lista de palabras)
    tokens = url.split()
    
    # Cargar stopwords (español e inglés suelen ser útiles para URLs)
    stop_words = set(stopwords.words('english'))
    
    # Lematización
    lemmatizer = WordNetLemmatizer()
    
    # Filtrar stopwords y lematizar cada token
    cleaned_tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    
    # Unir de nuevo en una cadena de texto (necesario para la mayoría de vectorizadores)
    return " ".join(cleaned_tokens)

[nltk_data] Downloading package stopwords to /home/vscode/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /home/vscode/nltk_data...
[nltk_data] Downloading package omw-1.4 to /home/vscode/nltk_data...


In [ ]:
# Aplicar la función a la columna de URLs
# (Asumiendo que tu columna se llama 'url')
df['cleaned_url'] = df['url'].apply(preprocess_url)

In [ ]:
# División del conjunto de datos (Train y Test)
# X: Las URLs procesadas | y: La columna is_spam (convertida a int)
X = df['cleaned_url']
y = df['is_spam'].astype(int)

# Dividimos con un 20% para test y un estado aleatorio para reproducibilidad
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Verificación rápida
print(f"Muestras de entrenamiento: {len(X_train)}")
print(f"Muestras de prueba: {len(X_test)}")
print("\nEjemplo de URL procesada:")
print(f"Original: {df['url'].iloc[0]}")
print(f"Limpia: {df['cleaned_url'].iloc[0]}")

Muestras de entrenamiento: 2399
Muestras de prueba: 600

Ejemplo de URL procesada:
Original: https://briefingday.us8.list-manage.com/unsubscribe
Limpia: http briefingday us8 list manage com unsubscribe


# 3. SVM

## 3.1 Vectorización

In [13]:
# Convertimos el texto limpio en una matriz de números
vectorizer = TfidfVectorizer()

# Ajustamos el vectorizador con los datos de entrenamiento y transformamos ambos sets
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

## 3.2 Construcción del SVM 

In [ ]:
# Inicializamos el modelo con parámetros por defecto
# Nota: SVC usa por defecto kernel='rbf'
svm_model = SVC()

# Entrenamos el modelo
svm_model.fit(X_train_vec, y_train)

,C,1.0
,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


## 3.3 Análisis de resultados

In [ ]:
# Realizamos las predicciones sobre el set de prueba
y_pred = svm_model.predict(X_test_vec)

# Mostramos métricas de rendimiento
print("--- RESULTADOS DEL SVM (PARÁMETROS POR DEFECTO) ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nReporte de Clasificación:")
print(classification_report(y_test, y_pred))

--- RESULTADOS DEL SVM (PARÁMETROS POR DEFECTO) ---
Accuracy: 0.9483

Reporte de Clasificación:
              precision    recall  f1-score   support

           0       0.96      0.97      0.97       455
           1       0.91      0.88      0.89       145

    accuracy                           0.95       600
   macro avg       0.93      0.92      0.93       600
weighted avg       0.95      0.95      0.95       600



# 4. Optimización del modelo

## 4.1 Definimos de los parámetros que queremos probar

In [ ]:
param_grid = {
    'C': [0.1, 1, 10, 100],              # Fuerza de regularización
    'gamma': [1, 0.1, 0.01, 0.001],      # Coeficiente del kernel (ancho de la campana)
    'kernel': ['linear', 'rbf', 'poly']  # Tipos de funciones de decisión
}

## 4.2 Inicializamos el buscador con validación cruzada (cv=5)

In [ ]:
# Esto dividirá el set de entrenamiento en 5 partes para probar cada combinación
grid = GridSearchCV(SVC(), param_grid, refit=True, verbose=2, cv=5)

## 4.3 Entrenamos el buscador 

In [ ]:
grid.fit(X_train_vec, y_train)

Fitting 5 folds for each of 48 candidates, totalling 240 fits
[CV] END ......................C=0.1, gamma=1, kernel=linear; total time=   0.2s
[CV] END ......................C=0.1, gamma=1, kernel=linear; total time=   0.2s
[CV] END ......................C=0.1, gamma=1, kernel=linear; total time=   0.2s
[CV] END ......................C=0.1, gamma=1, kernel=linear; total time=   0.2s
[CV] END ......................C=0.1, gamma=1, kernel=linear; total time=   0.2s
[CV] END .........................C=0.1, gamma=1, kernel=rbf; total time=   0.3s
[CV] END .........................C=0.1, gamma=1, kernel=rbf; total time=   0.3s
[CV] END .........................C=0.1, gamma=1, kernel=rbf; total time=   0.3s
[CV] END .........................C=0.1, gamma=1, kernel=rbf; total time=   0.3s
[CV] END .........................C=0.1, gamma=1, kernel=rbf; total time=   0.3s
[CV] END ........................C=0.1, gamma=1, kernel=poly; total time=   0.4s
[CV] END ........................C=0.1, gamma=1

,estimator,SVC()
,param_grid,"{'C': [0.1, 1, ...], 'gamma': [1, 0.1, ...], 'kernel': ['linear', 'rbf', ...]}"
,scoring,None
,n_jobs,None
,refit,True
,cv,5
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,C,10


## 4.4 Ver los mejores parámetros encontrados

In [ ]:
print("\n--- MEJORES PARÁMETROS ENCONTRADOS ---")
print(grid.best_params_)


--- MEJORES PARÁMETROS ENCONTRADOS ---
{'C': 10, 'gamma': 0.1, 'kernel': 'rbf'}


## 4.5 Evaluación del modelo optimizado

In [18]:
grid_predictions = grid.predict(X_test_vec)

print("\n--- RESULTADOS DEL SVM OPTIMIZADO ---")
print(f"Accuracy: {accuracy_score(y_test, grid_predictions):.4f}")
print("\nReporte de Clasificación Final:")
print(classification_report(y_test, grid_predictions))


--- RESULTADOS DEL SVM OPTIMIZADO ---
Accuracy: 0.9500

Reporte de Clasificación Final:
              precision    recall  f1-score   support

           0       0.98      0.96      0.97       455
           1       0.88      0.92      0.90       145

    accuracy                           0.95       600
   macro avg       0.93      0.94      0.93       600
weighted avg       0.95      0.95      0.95       600



# 5. Guardar modelo

In [19]:
# 1. Definimos la ruta
folder_path = '/workspaces/steven10015-intro-ml/models'

# 2. Creamos la carpeta si no existe
if not os.path.exists(folder_path):
    os.makedirs(folder_path)

# 3. Guardamos el modelo SVM optimizado
# 'wb' significa "write binary" (escritura binaria)
with open(os.path.join(folder_path, 'svm_model.pkl'), 'wb') as f:
    pickle.dump(grid.best_estimator_, f)

# 4. Guardamos el vectorizador
with open(os.path.join(folder_path, 'tfidf_vectorizer.pkl'), 'wb') as f:
    pickle.dump(vectorizer, f)

print(f"✅ Archivos .pkl generados con pickle en: {folder_path}")

✅ Archivos .pkl generados con pickle en: /workspaces/steven10015-intro-ml/models
